# 📊 Análise de Dados
## Crime Data from 2020 to Present - Los Angeles

Este notebook realiza análises exploratórias e gera visualizações para insights sobre os dados de crimes.

### Análises:
1. Análise Temporal de Crimes
2. Análise Geográfica
3. Análise por Tipo de Crime
4. Perfil das Vítimas
5. Correlações e Insights

In [ ]:
# Importações
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Importar módulos locais
import sys
sys.path.append('..')
from src.visualization import *

# Configurações
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

print('✅ Bibliotecas carregadas com sucesso!')

In [ ]:
# Carregar dados processados
df = pd.read_csv('../data/processed/crime_data_processed.csv')

# Converter colunas de data
df['DATE OCC'] = pd.to_datetime(df['DATE OCC'], errors='coerce')
df['Date Rptd'] = pd.to_datetime(df['Date Rptd'], errors='coerce')

print(f"Dataset carregado: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
df.head()

## 1. Análise Temporal de Crimes

In [ ]:
# Tendências temporais
fig = plot_temporal_trends(df, 'DATE OCC')
plt.show()

In [ ]:
# Crimes por ano
if 'YEAR' in df.columns:
    yearly_crimes = df.groupby('YEAR').size()
    
    plt.figure(figsize=(10, 6))
    bars = plt.bar(yearly_crimes.index.astype(str), yearly_crimes.values, color='steelblue')
    
    # Adicionar valores
    for bar, val in zip(bars, yearly_crimes.values):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000, 
                f'{val:,}', ha='center', fontsize=10)
    
    plt.xlabel('Ano')
    plt.ylabel('Número de Crimes')
    plt.title('Total de Crimes por Ano')
    plt.tight_layout()
    plt.show()

In [ ]:
# Análise por período do dia
if 'PERIOD' in df.columns:
    period_counts = df['PERIOD'].value_counts()
    
    plt.figure(figsize=(10, 6))
    colors = ['#2C3E50', '#E74C3C', '#F39C12', '#3498DB']
    plt.pie(period_counts.values, labels=period_counts.index, autopct='%1.1f%%', 
            colors=colors, explode=[0.02]*len(period_counts))
    plt.title('Distribuição de Crimes por Período do Dia')
    plt.show()

In [ ]:
# Crimes por fim de semana vs dias úteis
if 'IS_WEEKEND' in df.columns:
    weekend_counts = df['IS_WEEKEND'].map({0: 'Dia Útil', 1: 'Fim de Semana'}).value_counts()
    
    plt.figure(figsize=(8, 6))
    plt.bar(weekend_counts.index, weekend_counts.values, color=['#3498DB', '#E74C3C'])
    plt.xlabel('Tipo de Dia')
    plt.ylabel('Número de Crimes')
    plt.title('Crimes: Dia Útil vs Fim de Semana')
    
    for i, v in enumerate(weekend_counts.values):
        plt.text(i, v + 1000, f'{v:,}', ha='center')
    
    plt.show()

## 2. Análise Geográfica

In [ ]:
# Distribuição geográfica
fig = plot_geographic_distribution(df, sample_size=20000)
plt.show()

In [ ]:
# Top 10 áreas com mais crimes
fig = plot_crime_distribution(df, 'AREA NAME', top_n=10, 
                              title='Top 10 Áreas com Mais Crimes')
plt.show()

In [ ]:
# Mapa de calor por área e hora
if 'AREA NAME' in df.columns and 'HOUR' in df.columns:
    top_areas = df['AREA NAME'].value_counts().head(10).index
    df_top = df[df['AREA NAME'].isin(top_areas)]
    
    pivot = df_top.pivot_table(
        values='DR_NO', 
        index='AREA NAME', 
        columns='HOUR', 
        aggfunc='count'
    )
    
    plt.figure(figsize=(16, 8))
    sns.heatmap(pivot, cmap='YlOrRd')
    plt.xlabel('Hora do Dia')
    plt.ylabel('Área')
    plt.title('Frequência de Crimes por Área e Hora')
    plt.tight_layout()
    plt.show()

## 3. Análise por Tipo de Crime

In [ ]:
# Top 15 tipos de crimes
fig = plot_crime_distribution(df, 'Crm Cd Desc', top_n=15,
                              title='Top 15 Tipos de Crimes Mais Frequentes')
plt.show()

In [ ]:
# Crimes violentos vs não violentos
if 'IS_VIOLENT' in df.columns:
    violent_counts = df['IS_VIOLENT'].map({0: 'Não Violento', 1: 'Violento'}).value_counts()
    
    plt.figure(figsize=(8, 8))
    plt.pie(violent_counts.values, labels=violent_counts.index, autopct='%1.1f%%',
            colors=['#27AE60', '#C0392B'], explode=[0, 0.05])
    plt.title('Proporção de Crimes Violentos')
    plt.show()

In [ ]:
# Tipos de armas utilizadas
if 'Weapon Desc' in df.columns:
    weapons = df['Weapon Desc'].dropna()
    weapons = weapons[weapons != 'Unknown']
    
    fig = plot_crime_distribution(
        pd.DataFrame({'Weapon Desc': weapons}), 
        'Weapon Desc', 
        top_n=10,
        title='Top 10 Armas Utilizadas em Crimes'
    )
    plt.show()

In [ ]:
# Locais mais comuns de crimes
if 'Premis Desc' in df.columns:
    fig = plot_crime_distribution(df, 'Premis Desc', top_n=10,
                                  title='Top 10 Locais de Ocorrência de Crimes')
    plt.show()

## 4. Perfil das Vítimas

In [ ]:
# Análise do perfil das vítimas
fig = plot_victim_profile(df)
plt.show()

In [ ]:
# Crimes por faixa etária e tipo
if 'AGE_GROUP' in df.columns and 'IS_VIOLENT' in df.columns:
    age_violent = pd.crosstab(df['AGE_GROUP'], df['IS_VIOLENT'])
    age_violent.columns = ['Não Violento', 'Violento']
    
    age_violent.plot(kind='bar', figsize=(10, 6), color=['#27AE60', '#C0392B'])
    plt.xlabel('Faixa Etária')
    plt.ylabel('Número de Crimes')
    plt.title('Crimes por Faixa Etária e Violência')
    plt.legend(title='Tipo')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Distribuição por sexo e tipo de crime
if 'Vict Sex' in df.columns:
    top_crimes = df['Crm Cd Desc'].value_counts().head(5).index
    df_top_crimes = df[df['Crm Cd Desc'].isin(top_crimes)]
    
    cross_tab = pd.crosstab(df_top_crimes['Crm Cd Desc'], df_top_crimes['Vict Sex'])
    
    cross_tab.plot(kind='barh', figsize=(12, 6))
    plt.xlabel('Número de Crimes')
    plt.ylabel('Tipo de Crime')
    plt.title('Top 5 Crimes por Sexo da Vítima')
    plt.legend(title='Sexo')
    plt.tight_layout()
    plt.show()

## 5. Correlações e Insights

In [ ]:
# Matriz de correlação
fig = plot_correlation_matrix(df)
plt.show()

In [ ]:
# Status dos casos
if 'Status Desc' in df.columns:
    status_counts = df['Status Desc'].value_counts()
    
    plt.figure(figsize=(10, 6))
    plt.barh(status_counts.index, status_counts.values, color='teal')
    plt.xlabel('Número de Casos')
    plt.title('Status dos Casos de Crime')
    
    for i, v in enumerate(status_counts.values):
        plt.text(v + 1000, i, f'{v:,}', va='center')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Estatísticas resumidas
print("="*60)
print("RESUMO ESTATÍSTICO DO DATASET DE CRIMES")
print("="*60)
print(f"\n📊 Total de registros: {len(df):,}")
print(f"📅 Período: {df['DATE OCC'].min().strftime('%Y-%m-%d')} a {df['DATE OCC'].max().strftime('%Y-%m-%d')}")
print(f"🗺️ Número de áreas: {df['AREA NAME'].nunique()}")
print(f"🔢 Tipos de crimes: {df['Crm Cd Desc'].nunique()}")

if 'Vict Age' in df.columns:
    print(f"\n👤 Idade média das vítimas: {df['Vict Age'].mean():.1f} anos")

if 'IS_VIOLENT' in df.columns:
    pct_violent = df['IS_VIOLENT'].mean() * 100
    print(f"⚠️ Porcentagem de crimes violentos: {pct_violent:.1f}%")

print(f"\n🏆 Área com mais crimes: {df['AREA NAME'].value_counts().idxmax()}")
print(f"🔝 Crime mais comum: {df['Crm Cd Desc'].value_counts().idxmax()}")

if 'HOUR' in df.columns:
    print(f"⏰ Hora com mais crimes: {df['HOUR'].value_counts().idxmax()}:00")

## 💾 Salvando Visualizações

In [ ]:
import os
os.makedirs('../outputs/figures', exist_ok=True)

# Salvar principais visualizações
fig1 = plot_temporal_trends(df, 'DATE OCC')
save_figure(fig1, 'temporal_trends.png')
plt.close()

fig2 = plot_geographic_distribution(df, sample_size=20000)
save_figure(fig2, 'geographic_distribution.png')
plt.close()

fig3 = plot_victim_profile(df)
save_figure(fig3, 'victim_profile.png')
plt.close()

fig4 = plot_crime_distribution(df, 'Crm Cd Desc', top_n=15)
save_figure(fig4, 'crime_types.png')
plt.close()

print("\n✅ Todas as visualizações foram salvas!")

## 📊 Resumo da Análise de Dados

### Principais Insights:

1. **Padrões Temporais**
   - Crimes aumentam durante o período noturno
   - Variação sazonal ao longo do ano
   - Diferença entre dias úteis e fins de semana

2. **Distribuição Geográfica**
   - Concentração em áreas centrais de Los Angeles
   - Áreas com maior incidência identificadas

3. **Tipos de Crime**
   - Roubo de veículos é um dos crimes mais comuns
   - Proporção de crimes violentos analisada

4. **Perfil das Vítimas**
   - Idade média e distribuição por faixa etária
   - Diferenças por sexo e descendência

### Visualizações Salvas:
- `outputs/figures/temporal_trends.png`
- `outputs/figures/geographic_distribution.png`
- `outputs/figures/victim_profile.png`
- `outputs/figures/crime_types.png`